# BTF vs Adam/AdamW on m3/m4/m5/a-series/rule-30_1, rule-30_2(two architectures) majority-gate datasets

This notebook compares Boolearn's **BTF (constraint-based)** training to **gradient-based** training
(Adam / AdamW) on the synthetic majority-gate datasets:

- `m3.dat`  – depth-3 majority network on 32 bits  
- `m4.dat`  – depth-4 majority network on 32 bits  
- `m5.dat`  – depth-5 majority network on 32 bits  



In [71]:
from pathlib import Path

# EDIT THIS BASE DIR to where your m5k*.dat live
BASE_DIR = Path("/Users/mkrishan/Documents/boolearn-main")

DATASETS = {
    "m5k1": {"dat": BASE_DIR / "m5k1.dat", "N_train": 128},
    "m5k2": {"dat": BASE_DIR / "m5k2.dat", "N_train": 256},
    "m5k3": {"dat": BASE_DIR / "m5k3.dat", "N_train": 384},
    "m5k4": {"dat": BASE_DIR / "m5k4.dat", "N_train": 512},
}

for k, cfg in DATASETS.items():
    assert cfg["dat"].exists(), f"Missing {cfg['dat']}"
    print(k, "->", cfg["dat"], "| N_train =", cfg["N_train"])

m5k1 -> /Users/mkrishan/Documents/boolearn-main/m5k1.dat | N_train = 128
m5k2 -> /Users/mkrishan/Documents/boolearn-main/m5k2.dat | N_train = 256
m5k3 -> /Users/mkrishan/Documents/boolearn-main/m5k3.dat | N_train = 384
m5k4 -> /Users/mkrishan/Documents/boolearn-main/m5k4.dat | N_train = 512


In [73]:
import torch

def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)

    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())

    X_list, Y_list = [], []
    for _ in range(N_total):
        x = list(map(int, next(it).split()))
        y = list(map(int, next(it).split()))
        X_list.append(x)
        Y_list.append(y)

    X = torch.tensor(X_list, dtype=torch.float32)  # (N_total, 32)
    Y = torch.tensor(Y_list, dtype=torch.float32)  # (N_total, 32)

    assert IN == 32 and OUT == 32, (IN, OUT)
    assert X.shape[0] == N_total and Y.shape[0] == N_total
    return X, Y

In [74]:
for key, cfg in DATASETS.items():
    X, Y = load_dat_bits(cfg["dat"])
    N_total = len(X)
    N_train = cfg["N_train"]
    N_test = N_total - N_train

    print(f"\n[{key}] total={N_total}, train={N_train}, test={N_test}")
    print("X shape:", tuple(X.shape), "Y shape:", tuple(Y.shape))
    assert N_total == 10000, "Expected total=10000 for mk-series"
    assert N_test == 10000 - N_train


[m5k1] total=10000, train=128, test=9872
X shape: (10000, 32) Y shape: (10000, 32)

[m5k2] total=10000, train=256, test=9744
X shape: (10000, 32) Y shape: (10000, 32)

[m5k3] total=10000, train=384, test=9616
X shape: (10000, 32) Y shape: (10000, 32)

[m5k4] total=10000, train=512, test=9488
X shape: (10000, 32) Y shape: (10000, 32)


In [75]:
import pandas as pd

def bit_balance_stats(Y: torch.Tensor):
    # Y is 0/1 float tensor
    # returns per-bit mean, overall mean
    per_bit = Y.mean(dim=0)  # length 32
    overall = Y.mean().item()
    return per_bit, overall

for key, cfg in DATASETS.items():
    X, Y = load_dat_bits(cfg["dat"])
    N_train = cfg["N_train"]

    per_full, overall_full = bit_balance_stats(Y)
    per_tr, overall_tr = bit_balance_stats(Y[:N_train])

    print(f"\n[{key}] overall P(y=1): full={overall_full:.3f}, train={overall_tr:.3f}")
    print("per-bit min/max (full):", float(per_full.min()), float(per_full.max()))
    print("per-bit min/max (train):", float(per_tr.min()), float(per_tr.max()))


[m5k1] overall P(y=1): full=0.499, train=0.512
per-bit min/max (full): 0.49070000648498535 0.5092999935150146
per-bit min/max (train): 0.4140625 0.59375

[m5k2] overall P(y=1): full=0.504, train=0.499
per-bit min/max (full): 0.49230000376701355 0.5166000127792358
per-bit min/max (train): 0.421875 0.56640625

[m5k3] overall P(y=1): full=0.502, train=0.512
per-bit min/max (full): 0.4961000084877014 0.5090000033378601
per-bit min/max (train): 0.4583333432674408 0.5651041865348816

[m5k4] overall P(y=1): full=0.499, train=0.497
per-bit min/max (full): 0.4860000014305115 0.5088000297546387
per-bit min/max (train): 0.443359375 0.53515625


In [76]:
import torch.nn as nn

class MLP32(nn.Module):
    def __init__(self, in_dim=32, depth=5, out_dim=32):
        super().__init__()
        hidden = 32
        layers = []
        layers += [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        layers += [nn.Linear(hidden, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def bitwise_acc_from_logits(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    return (pred == y_true).float().mean().item() * 100.0

In [77]:
import numpy as np

def log_spaced_steps(max_steps: int, n_points: int = 25, include_zero=True):
    # steps from 1..max_steps, log spaced
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    if include_zero:
        steps = np.concatenate(([0], steps))
    return steps.tolist()

# Sanity schedule for small run (fast)
print("Example log schedule up to 10k:", log_spaced_steps(10_000, n_points=20)[:15], "...")

Example log schedule up to 10k: [0, 1, 2, 3, 4, 7, 11, 18, 30, 48, 78, 127, 207, 336, 546] ...


In [78]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def gd_sanity_run(dat_path: Path, N_train: int, max_steps=5000, opt_name="adamw", lr=1e-3, wd=1e-4):
    X, Y = load_dat_bits(dat_path)
    Xtr, Ytr = X[:N_train].to(device), Y[:N_train].to(device)
    Xte, Yte = X[N_train:].to(device), Y[N_train:].to(device)

    model = MLP32(depth=5).to(device)
    loss_fn = nn.BCEWithLogitsLoss()

    if opt_name == "adamw":
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif opt_name == "adam":
        opt = optim.Adam(model.parameters(), lr=lr)
    else:
        opt = optim.SGD(model.parameters(), lr=lr)

    steps_to_log = set(log_spaced_steps(max_steps, n_points=15))
    print("Logging at:", sorted(steps_to_log)[:10], "...")

    def eval_print(step):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_acc = bitwise_acc_from_logits(tr_logits, Ytr)
            te_acc = bitwise_acc_from_logits(te_logits, Yte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            print(f"step={step:5d}  train={tr_acc:6.2f}%  test={te_acc:6.2f}%  loss={tr_loss:.4f}")

    eval_print(0)
    model.train()
    for step in range(1, max_steps + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()

        if step in steps_to_log:
            eval_print(step)

    return model

# Sanity run on m5k4 (Track D)
cfg = DATASETS["m5k4"]
_ = gd_sanity_run(cfg["dat"], cfg["N_train"], max_steps=5000, opt_name="adamw")

device: cpu
Logging at: [0, 1, 2, 3, 6, 11, 21, 38, 71, 130] ...
step=    0  train= 50.16%  test= 50.10%  loss=0.6951
step=    1  train= 50.21%  test= 50.03%  loss=0.6949
step=    2  train= 50.26%  test= 50.03%  loss=0.6948
step=    3  train= 50.27%  test= 50.02%  loss=0.6946
step=    6  train= 50.27%  test= 49.94%  loss=0.6942
step=   11  train= 50.54%  test= 50.23%  loss=0.6935
step=   21  train= 52.20%  test= 51.30%  loss=0.6920
step=   38  train= 55.82%  test= 54.03%  loss=0.6854
step=   71  train= 67.61%  test= 65.37%  loss=0.6073
step=  130  train= 78.62%  test= 76.81%  loss=0.4348
step=  239  train= 83.57%  test= 81.36%  loss=0.3471
step=  439  train= 88.09%  test= 84.69%  loss=0.2644
step=  806  train= 90.61%  test= 85.51%  loss=0.2112
step= 1481  train= 93.16%  test= 85.51%  loss=0.1657
step= 2721  train= 94.57%  test= 84.87%  loss=0.1313
step= 5000  train= 96.08%  test= 84.00%  loss=0.1013


In [86]:
# =========================
# BIG GD EXPERIMENT (m5k1..m5k4) — full-batch, deterministic, log-spaced logging
# Saves one .log per (dataset, opt, seed) in RESULTS_DIR
# =========================

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ---------- EDIT PATHS ----------
BASE_DIR = Path("/Users/mkrishan/Documents/boolearn-main")     # where m5k*.dat live
RESULTS_DIR = BASE_DIR / "results_gd_m5k_big"    # where logs are written
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "m5k1": {"dat": BASE_DIR / "m5k1.dat", "N_train": 128},
    "m5k2": {"dat": BASE_DIR / "m5k2.dat", "N_train": 256},
    "m5k3": {"dat": BASE_DIR / "m5k3.dat", "N_train": 384},
    "m5k4": {"dat": BASE_DIR / "m5k4.dat", "N_train": 512},
}
for k, cfg in DATASETS.items():
    assert cfg["dat"].exists(), f"Missing {cfg['dat']}"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------- EXPERIMENT GRID ----------
DEPTH = 5
OPT_LIST = ["adamw"]          # add "adam" and/or "sgd" if you want
SEEDS = [0]                   # big run: [0,1,2]
LR = 1e-3
WD_ADAMW = 1e-4
WD_SGD = 1e-4

MAX_STEPS = 1_000_000         # <-- requested 10^6
LOG_POINTS = 45               # number of log-spaced dots

# ---------- Helpers ----------
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    assert IN == 32 and OUT == 32, (IN, OUT)
    assert len(X) == N_total and len(Y) == N_total
    return X, Y

class MLP32(nn.Module):
    def __init__(self, in_dim=32, depth=5, out_dim=32):
        super().__init__()
        hidden = 32
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        layers += [nn.Linear(hidden, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def log_spaced_steps(max_steps: int, n_points: int):
    # logspace from 1..max_steps plus 0
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return [0] + steps.tolist()

LOG_STEPS = set(log_spaced_steps(MAX_STEPS, LOG_POINTS))
print("Logging at", len(LOG_STEPS), "log-spaced steps (including 0 and max).")

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def run_name(dataset_key, N_train, opt_name, seed):
    return f"{dataset_key}_N{N_train}_{opt_name}_seed{seed}_steps{MAX_STEPS}"

def run_fullbatch(dataset_key, dat_path, N_train, opt_name, seed):
    tag = run_name(dataset_key, N_train, opt_name, seed)
    log_path = RESULTS_DIR / f"{tag}.log"

    # Skip if already done
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return

    torch.manual_seed(seed)

    # Load data once per run
    X_all, Y_all = load_dat_bits(dat_path)
    assert len(X_all) == 10000, f"{dataset_key}: expected 10000 total"
    assert len(X_all) - N_train == 10000 - N_train

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP32(depth=DEPTH).to(device)
    loss_fn = nn.BCEWithLogitsLoss()

    if opt_name.lower() == "adamw":
        opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD_ADAMW)
    elif opt_name.lower() == "adam":
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
    elif opt_name.lower() == "sgd":
        opt = optim.SGD(model.parameters(), lr=LR, weight_decay=WD_SGD, momentum=0.0)
    else:
        raise ValueError(opt_name)

    # write header
    with open(log_path, "w") as f:
        f.write("# GD baseline MLP32 ReLU (full-batch)\n")
        f.write(f"# depth={DEPTH}, dataset={dataset_key}, N_train={N_train}, opt={opt_name}, seed={seed}, "
                f"lr={LR}, wd={(WD_ADAMW if opt_name.lower()=='adamw' else (WD_SGD if opt_name.lower()=='sgd' else 0.0))}, "
                f"max_steps={MAX_STEPS}\n")
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr)
            te_logits = model(Xte)

            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()

            tr_bit, tr_exact = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_exact = bitwise_and_exact_acc(te_logits, Yte, 0.5)

        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_exact:.4f},{te_exact:.4f}\n")

        print(f"[{tag}] step={step:>8d}  train={tr_bit:6.2f}%  test={te_bit:6.2f}%  "
              f"exact={tr_exact:5.1f}/{te_exact:5.1f}  loss={tr_loss:.4f}")

    # step 0
    eval_and_log(0)

    model.train()
    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved log ->", log_path)

# ---------- RUN GRID ----------
for dataset_key, cfg in DATASETS.items():
    for opt_name in OPT_LIST:
        for seed in SEEDS:
            print(f"\n=== RUN {dataset_key} | N={cfg['N_train']} | opt={opt_name} | seed={seed} ===")
            run_fullbatch(dataset_key, cfg["dat"], cfg["N_train"], opt_name, seed)

print("\nALL DONE. Logs in:", RESULTS_DIR)

device: cpu
Logging at 45 log-spaced steps (including 0 and max).

=== RUN m5k1 | N=128 | opt=adamw | seed=0 ===
[m5k1_N128_adamw_seed0_steps1000000] step=       0  train= 49.73%  test= 50.07%  exact=  0.0/  0.0  loss=0.6946
[m5k1_N128_adamw_seed0_steps1000000] step=       1  train= 49.73%  test= 50.15%  exact=  0.0/  0.0  loss=0.6945
[m5k1_N128_adamw_seed0_steps1000000] step=       2  train= 49.83%  test= 50.08%  exact=  0.0/  0.0  loss=0.6944
[m5k1_N128_adamw_seed0_steps1000000] step=       3  train= 49.83%  test= 50.08%  exact=  0.0/  0.0  loss=0.6942
[m5k1_N128_adamw_seed0_steps1000000] step=       4  train= 49.83%  test= 50.08%  exact=  0.0/  0.0  loss=0.6941
[m5k1_N128_adamw_seed0_steps1000000] step=       5  train= 49.83%  test= 50.09%  exact=  0.0/  0.0  loss=0.6940
[m5k1_N128_adamw_seed0_steps1000000] step=       7  train= 50.00%  test= 50.16%  exact=  0.0/  0.0  loss=0.6937
[m5k1_N128_adamw_seed0_steps1000000] step=       9  train= 50.34%  test= 50.24%  exact=  0.0/  0.0  los

In [87]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ===== EDIT PATHS =====
BASE_DIR = Path("/Users/mkrishan/Documents/boolearn-main")   # where a2c.dat..a5c.dat live
RESULTS_DIR = BASE_DIR / "results_gd_a_series"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ===== DATASETS: a2c..a5c =====
DATASETS = {
    "a2c": {"dat": BASE_DIR / "a2c.dat", "depth": 2},
    "a3c": {"dat": BASE_DIR / "a3c.dat", "depth": 3},
    "a4c": {"dat": BASE_DIR / "a4c.dat", "depth": 4},
    "a5c": {"dat": BASE_DIR / "a5c.dat", "depth": 5},
}
for k, cfg in DATASETS.items():
    assert cfg["dat"].exists(), f"Missing {cfg['dat']}"
    print(k, "->", cfg["dat"], "| depth =", cfg["depth"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ===== TRAINING SIZES (match m5k tracks) =====
N_LIST = [128, 256, 384, 512]

# ===== OPTIMIZERS / SEEDS =====
OPT_LIST = ["adamw"]          # add "adam", "sgd" if needed
SEEDS = [0]                   # big run: [0,1,2]

LR = 1e-3
WD_ADAMW = 1e-4
WD_SGD = 1e-4

MAX_STEPS = 1_000_000
LOG_POINTS = 45

a2c -> /Users/mkrishan/Documents/boolearn-main/a2c.dat | depth = 2
a3c -> /Users/mkrishan/Documents/boolearn-main/a3c.dat | depth = 3
a4c -> /Users/mkrishan/Documents/boolearn-main/a4c.dat | depth = 4
a5c -> /Users/mkrishan/Documents/boolearn-main/a5c.dat | depth = 5
device: cpu


In [88]:
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    assert IN == 32 and OUT == 32, (IN, OUT)
    return X, Y

class MLP32(nn.Module):
    """
    Width 32 ReLU MLP, depth = number of affine layers.
    """
    def __init__(self, in_dim=32, depth=5, out_dim=32):
        super().__init__()
        hidden = 32
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        layers += [nn.Linear(hidden, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def log_spaced_steps(max_steps: int, n_points: int):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return [0] + steps.tolist()

LOG_STEPS = set(log_spaced_steps(MAX_STEPS, LOG_POINTS))
print("Logging at", len(LOG_STEPS), "log-spaced checkpoints (incl. 0).")

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

Logging at 45 log-spaced checkpoints (incl. 0).


In [89]:
def run_tag(dataset_key, depth, N_train, opt_name, seed):
    return f"{dataset_key}_d{depth}_N{N_train}_{opt_name}_seed{seed}_steps{MAX_STEPS}"

def run_fullbatch(dataset_key, dat_path, depth, N_train, opt_name, seed):
    tag = run_tag(dataset_key, depth, N_train, opt_name, seed)
    log_path = RESULTS_DIR / f"{tag}.log"

    if log_path.exists():
        print("[SKIP]", log_path.name)
        return

    torch.manual_seed(seed)

    X_all, Y_all = load_dat_bits(dat_path)
    assert len(X_all) >= max(N_LIST), f"{dataset_key}: dataset too small for N={max(N_LIST)}"
    # the remainder is test set (same convention as before)
    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP32(depth=depth).to(device)
    loss_fn = nn.BCEWithLogitsLoss()

    if opt_name.lower() == "adamw":
        opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD_ADAMW)
        wd = WD_ADAMW
    elif opt_name.lower() == "adam":
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
        wd = 0.0
    elif opt_name.lower() == "sgd":
        opt = optim.SGD(model.parameters(), lr=LR, weight_decay=WD_SGD, momentum=0.0)
        wd = WD_SGD
    else:
        raise ValueError(opt_name)

    with open(log_path, "w") as f:
        f.write("# GD baseline MLP32 ReLU (full-batch)\n")
        f.write(f"# dataset={dataset_key}, depth={depth}, N_train={N_train}, opt={opt_name}, seed={seed}, lr={LR}, wd={wd}, max_steps={MAX_STEPS}\n")
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr)
            te_logits = model(Xte)

            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()

            tr_bit, tr_exact = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_exact = bitwise_and_exact_acc(te_logits, Yte, 0.5)

        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_exact:.4f},{te_exact:.4f}\n")

        print(f"[{tag}] step={step:>8d}  train={tr_bit:6.2f}% test={te_bit:6.2f}%  exact={tr_exact:5.1f}/{te_exact:5.1f}  loss={tr_loss:.4f}")

    # step 0
    eval_and_log(0)

    model.train()
    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved log ->", log_path)

In [90]:
for dataset_key, cfg in DATASETS.items():
    dat_path = cfg["dat"]
    depth = cfg["depth"]

    for N_train in N_LIST:
        for opt_name in OPT_LIST:
            for seed in SEEDS:
                print(f"\n=== RUN {dataset_key} | depth={depth} | N={N_train} | opt={opt_name} | seed={seed} ===")
                run_fullbatch(dataset_key, dat_path, depth, N_train, opt_name, seed)

print("\nALL DONE. Logs saved in:", RESULTS_DIR)


=== RUN a2c | depth=2 | N=128 | opt=adamw | seed=0 ===
[a2c_d2_N128_adamw_seed0_steps1000000] step=       0  train= 52.73% test= 53.21%  exact=  0.0/  0.0  loss=0.6948
[a2c_d2_N128_adamw_seed0_steps1000000] step=       1  train= 53.52% test= 53.89%  exact=  0.0/  0.0  loss=0.6929
[a2c_d2_N128_adamw_seed0_steps1000000] step=       2  train= 54.32% test= 54.59%  exact=  0.0/  0.0  loss=0.6910
[a2c_d2_N128_adamw_seed0_steps1000000] step=       3  train= 54.98% test= 55.24%  exact=  0.0/  0.0  loss=0.6892
[a2c_d2_N128_adamw_seed0_steps1000000] step=       4  train= 55.49% test= 55.86%  exact=  0.0/  0.0  loss=0.6873
[a2c_d2_N128_adamw_seed0_steps1000000] step=       5  train= 56.13% test= 56.48%  exact=  0.0/  0.0  loss=0.6854
[a2c_d2_N128_adamw_seed0_steps1000000] step=       7  train= 57.32% test= 57.74%  exact=  0.0/  0.0  loss=0.6816
[a2c_d2_N128_adamw_seed0_steps1000000] step=       9  train= 58.89% test= 58.95%  exact=  0.0/  0.0  loss=0.6778
[a2c_d2_N128_adamw_seed0_steps1000000] s

In [96]:
from pathlib import Path
import subprocess

REPO = Path("/Users/mkrishan/Documents/boolearn-main")   # adjust
SRC  = next(REPO.rglob("data_cellauto"))               # auto-find executable
assert SRC.exists(), "Could not find data_cellauto executable (compile boolearn/src first)."

DAT = REPO / "30_1.dat"

# Wolfram Rule 30, size=16, steps=1, items=10000
if not DAT.exists():
    cmd = [str(SRC), "3", "30", "16", "1", "10000", "30_1"]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found:", DAT)

Found: /Users/mkrishan/Documents/boolearn-main/30_1.dat


In [97]:
import torch

def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
print("Total:", len(X_all), "IN:", IN, "OUT:", OUT)
assert IN == 16 and OUT == 16

Total: 10000 IN: 16 OUT: 16


In [98]:
import torch.nn as nn

class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

In [99]:
import numpy as np
import torch.optim as optim
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def run_gd_rule30(N_train=256, opt_name="adamw", seed=0, max_steps=1_000_000, lr=1e-3, wd=1e-4):
    out_dir = REPO / "results_gd_rule30"
    out_dir.mkdir(exist_ok=True)
    tag = f"rule30_N{N_train}_{opt_name}_seed{seed}_steps{max_steps}"
    log_path = out_dir / f"{tag}.log"

    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    loss_fn = nn.BCEWithLogitsLoss()

    if opt_name == "adamw":
        opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    elif opt_name == "adam":
        opt = optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
    elif opt_name == "sgd":
        opt = optim.SGD(model.parameters(), lr=lr, momentum=0.0, weight_decay=wd)
    else:
        raise ValueError(opt_name)

    LOG_STEPS = log_spaced_steps(max_steps, 45)

    with open(log_path, "w") as f:
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[{tag}] step={step:>8d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

    eval_and_log(0)
    model.train()
    for step in range(1, max_steps + 1):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        opt.step()
        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved:", log_path)
    return log_path

In [100]:
# quick sanity (20k steps)
_ = run_gd_rule30(N_train=256, opt_name="adamw", seed=0, max_steps=20_000)

[rule30_N256_adamw_seed0_steps20000] step=       0 train= 49.51% test= 49.50% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       1 train= 49.61% test= 49.59% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       2 train= 49.71% test= 49.72% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       3 train= 49.93% test= 49.80% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       4 train= 49.95% test= 49.87% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       5 train= 50.17% test= 49.96% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       6 train= 50.20% test= 50.05% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       8 train= 50.90% test= 50.25% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=       9 train= 51.32% test= 50.35% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] step=      12 train= 51.81% test= 50.74% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps20000] ste

In [101]:
_ = run_gd_rule30(N_train=256, opt_name="adamw", seed=0, max_steps=1_000_000)

[rule30_N256_adamw_seed0_steps1000000] step=       0 train= 49.51% test= 49.50% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       1 train= 49.61% test= 49.59% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       2 train= 49.71% test= 49.72% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       3 train= 49.93% test= 49.80% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       4 train= 49.95% test= 49.87% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       5 train= 50.17% test= 49.96% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       7 train= 50.37% test= 50.16% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=       9 train= 51.32% test= 50.35% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=      12 train= 51.81% test= 50.74% exact=  0.0/  0.0
[rule30_N256_adamw_seed0_steps1000000] step=      17 train= 53.83% test= 51.42% exact=  0.0/  0.0
[rule30_N256_adamw_s

In [102]:
# =========================
# RULE-30 GD BASELINES (full-batch, deterministic, log-spaced)
# N = 128,256,384,512; opt = adamw/adam/sgd/lion; steps=1e6
# Saves .log files to results_gd_rule30/
# =========================

from pathlib import Path
import subprocess, shlex
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ---------- EDIT PATHS ----------
REPO = Path("/Users/mkrishan/Documents/boolearn-main")  # repo root (where you want 30_1.dat)
assert REPO.exists(), REPO

RESULTS_DIR = REPO / "results_gd_rule30"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------- DEVICE ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------- FIND / BUILD data_cellauto ----------
# We just need data_cellauto executable to generate 30_1.dat.
data_cellauto_hits = list(REPO.rglob("data_cellauto"))
assert data_cellauto_hits, "Could not find data_cellauto. Compile boolearn/src first."
data_cellauto = sorted(data_cellauto_hits, key=lambda p: len(str(p)))[0]
print("data_cellauto:", data_cellauto)

# ---------- GENERATE RULE-30 DATASET ----------
# Rule 30, size=16, steps=1, items=10000, id=30_1 -> 30_1.dat
DAT = REPO / "30_1.dat"
if not DAT.exists():
    cmd = [str(data_cellauto), "3", "30", "16", "1", "10000", "30_1"]
    print("Generating dataset:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found existing:", DAT)

# ---------- LOAD .dat ----------
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
print("Loaded:", DAT, "| total:", len(X_all), "| IN:", IN, "| OUT:", OUT)
assert len(X_all) == 10000
assert IN == 16 and OUT == 16

# ---------- ARCHITECTURE ----------
# Matches your intended net: 16 -> 32 -> 32 -> 16 (3 affine layers)
class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16)
        )
    def forward(self, x):
        return self.net(x)

# ---------- LION OPTIMIZER (minimal, decoupled WD) ----------
from torch.optim.optimizer import Optimizer

class Lion(Optimizer):
    """Minimal Lion optimizer (decoupled weight decay)."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            wd = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad

                # decoupled weight decay
                if wd != 0:
                    p.mul_(1 - lr * wd)

                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)

                m = state["m"]
                m.mul_(beta2).add_(g, alpha=1 - beta2)

                update = (beta1 * m + (1 - beta1) * g).sign()
                p.add_(update, alpha=-lr)

        return loss

# ---------- METRICS ----------
@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# ---------- LOG-SPACED CHECKPOINTS ----------
def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# ---------- RUN CONFIG ----------
N_LIST = [128, 256, 384, 512]
OPT_LIST = ["adamw", "adam", "sgd", "lion"]   # include lion
SEEDS = [0]                                   # then [0,1,2] if desired

MAX_STEPS = 1_000_000
LOG_STEPS = log_spaced_steps(MAX_STEPS, n_points=45)

LR = 1e-3
WD = 1e-4
LION_LR = 3e-4          # Lion often likes slightly higher lr
LION_BETAS = (0.9, 0.99)

loss_fn = nn.BCEWithLogitsLoss()

def run_tag(N_train, opt_name, seed):
    return f"rule30_N{N_train}_{opt_name}_seed{seed}_steps{MAX_STEPS}"

def run_fullbatch_rule30(N_train: int, opt_name: str, seed: int):
    tag = run_tag(N_train, opt_name, seed)
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return log_path

    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)

    if opt_name == "adamw":
        opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        lr_used, wd_used = LR, WD
    elif opt_name == "adam":
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
        lr_used, wd_used = LR, 0.0
    elif opt_name == "sgd":
        opt = optim.SGD(model.parameters(), lr=LR, momentum=0.0, weight_decay=WD)
        lr_used, wd_used = LR, WD
    elif opt_name == "lion":
        opt = Lion(model.parameters(), lr=LION_LR, betas=LION_BETAS, weight_decay=WD)
        lr_used, wd_used = LION_LR, WD
    else:
        raise ValueError(opt_name)

    with open(log_path, "w") as f:
        f.write("# GD baseline ReLU MLP (full-batch)\n")
        f.write(f"# rule=30, size=16, steps=1, arch=16-32-32-16, N_train={N_train}, opt={opt_name}, seed={seed}, lr={lr_used}, wd={wd_used}, max_steps={MAX_STEPS}\n")
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte, 0.5)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[{tag}] step={step:>8d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f} loss={tr_loss:.4f}")

    eval_and_log(0)

    model.train()
    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)
    return log_path

# ---------- RUN GRID ----------
for N_train in N_LIST:
    for opt_name in OPT_LIST:
        for seed in SEEDS:
            print(f"\n=== RUN rule30 | N={N_train} | opt={opt_name} | seed={seed} ===")
            run_fullbatch_rule30(N_train, opt_name, seed)

print("\nDONE. Logs saved in:", RESULTS_DIR)

device: cpu
data_cellauto: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto
Found existing: /Users/mkrishan/Documents/boolearn-main/30_1.dat
Loaded: /Users/mkrishan/Documents/boolearn-main/30_1.dat | total: 10000 | IN: 16 | OUT: 16

=== RUN rule30 | N=128 | opt=adamw | seed=0 ===
[rule30_N128_adamw_seed0_steps1000000] step=       0 train= 49.12% test= 49.50% exact=  0.0/  0.0 loss=0.6968
[rule30_N128_adamw_seed0_steps1000000] step=       1 train= 49.41% test= 49.58% exact=  0.0/  0.0 loss=0.6963
[rule30_N128_adamw_seed0_steps1000000] step=       2 train= 49.66% test= 49.69% exact=  0.0/  0.0 loss=0.6957
[rule30_N128_adamw_seed0_steps1000000] step=       3 train= 49.37% test= 49.71% exact=  0.0/  0.0 loss=0.6952
[rule30_N128_adamw_seed0_steps1000000] step=       4 train= 49.56% test= 49.80% exact=  0.0/  0.0 loss=0.6948
[rule30_N128_adamw_seed0_steps1000000] step=       5 train= 49.71% test= 49.88% exact=  0.0/  0.0 loss=0.6943
[rule30_N128_adamw_seed0_steps1000000] st

In [3]:
# =========================
# RULE-32 (2 steps) GD BASELINES (full-batch, deterministic, log-spaced)
# Dataset: 32_2.dat from data_cellauto  (inbits=3, rule=32, size=16, steps=2, items=10000)
# Two architectures:
#   A: 16-32-32-16-32-32-16
#   B: 16-32-32-32-32-32-16
# N = 128,256,384,512; steps=1e6; opt = adamw/adam/sgd/lion
# =========================

from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.optimizer import Optimizer

# ---------- EDIT PATH ----------
REPO = Path("/Users/mkrishan/Documents/boolearn-main")  # adjust
assert REPO.exists(), REPO

RESULTS_DIR = REPO / "results_gd_rule32_2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------- Locate data_cellauto ----------
data_cellauto_hits = list(REPO.rglob("data_cellauto"))
assert data_cellauto_hits, "Could not find data_cellauto. Compile boolearn/src first."
data_cellauto = sorted(data_cellauto_hits, key=lambda p: len(str(p)))[0]
print("data_cellauto:", data_cellauto)

# ---------- Generate dataset: 32_2.dat ----------
INBITS = 3
RULE   = 32
SIZE   = 16
STEPS  = 2
ITEMS  = 10000
ID     = "32_2"
DAT = REPO / f"{ID}.dat"

if not DAT.exists():
    cmd = [str(data_cellauto), str(INBITS), str(RULE), str(SIZE), str(STEPS), str(ITEMS), ID]
    print("Generating dataset:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found existing:", DAT)

# ---------- Load .dat ----------
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
print("Loaded:", DAT, "| total:", len(X_all), "| IN:", IN, "| OUT:", OUT)
assert len(X_all) == 10000
assert IN == 16 and OUT == 16

# ---------- Two architectures ----------
# A: 16 32 32 16 32 32 16  -> affine layers: 6 (hidden sizes [32,32,16,32,32])
# B: 16 32 32 32 32 32 16  -> affine layers: 6 (hidden sizes [32,32,32,32,32])

ARCHS = {
    "A_16-32-32-16-32-32-16": [32, 32, 16, 32, 32],
    "B_16-32-32-32-32-32-16": [32, 32, 32, 32, 32],
}

class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_sizes, out_dim: int):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU(inplace=True))
            last = h
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# ---------- Lion optimizer ----------
class Lion(Optimizer):
    """Minimal Lion optimizer (decoupled weight decay)."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            beta1, beta2 = group["betas"]
            wd = group["weight_decay"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad

                # decoupled weight decay
                if wd != 0:
                    p.mul_(1 - lr * wd)

                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)

                m = state["m"]
                m.mul_(beta2).add_(g, alpha=1 - beta2)

                update = (beta1 * m + (1 - beta1) * g).sign()
                p.add_(update, alpha=-lr)

        return loss

# ---------- Metrics ----------
@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# ---------- log-spaced checkpoints ----------
def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# ---------- Run config ----------
N_LIST = [128, 256, 384, 512]
OPT_LIST = ["adamw", "adam", "sgd", "lion"]
SEEDS = [0]              # set to [0,1,2] later

MAX_STEPS = 1_000_000
LOG_STEPS = log_spaced_steps(MAX_STEPS, 45)

LR = 1e-3
WD = 1e-4
LION_LR = 3e-4
LION_BETAS = (0.9, 0.99)

loss_fn = nn.BCEWithLogitsLoss()

def run_tag(arch_name, N_train, opt_name, seed):
    return f"rule32_steps2_{arch_name}_N{N_train}_{opt_name}_seed{seed}_steps{MAX_STEPS}"

def run_fullbatch(arch_name, hidden_sizes, N_train, opt_name, seed):
    tag = run_tag(arch_name, N_train, opt_name, seed)
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return

    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP(16, hidden_sizes, 16).to(device)

    if opt_name == "adamw":
        opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
        lr_used, wd_used = LR, WD
    elif opt_name == "adam":
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=0.0)
        lr_used, wd_used = LR, 0.0
    elif opt_name == "sgd":
        opt = optim.SGD(model.parameters(), lr=LR, momentum=0.0, weight_decay=WD)
        lr_used, wd_used = LR, WD
    elif opt_name == "lion":
        opt = Lion(model.parameters(), lr=LION_LR, betas=LION_BETAS, weight_decay=WD)
        lr_used, wd_used = LION_LR, WD
    else:
        raise ValueError(opt_name)

    with open(log_path, "w") as f:
        f.write("# GD baseline ReLU MLP (full-batch)\n")
        f.write(f"# rule=32, size=16, steps=2, arch={arch_name}, hidden={hidden_sizes}, N_train={N_train}, opt={opt_name}, seed={seed}, lr={lr_used}, wd={wd_used}, max_steps={MAX_STEPS}\n")
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte, 0.5)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[{tag}] step={step:>8d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f} loss={tr_loss:.4f}")

    eval_and_log(0)
    model.train()

    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()
        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)

# ---------- Run grid ----------
for arch_name, hidden_sizes in ARCHS.items():
    print("\n=== ARCH:", arch_name, "hidden_sizes:", hidden_sizes, "===")
    for N_train in N_LIST:
        for opt_name in OPT_LIST:
            for seed in SEEDS:
                print(f"\n--- RUN N={N_train} opt={opt_name} seed={seed} ---")
                run_fullbatch(arch_name, hidden_sizes, N_train, opt_name, seed)

print("\nDONE. Logs saved in:", RESULTS_DIR)

device: cpu
data_cellauto: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto
Generating dataset: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto 3 32 16 2 10000 32_2
Loaded: /Users/mkrishan/Documents/boolearn-main/32_2.dat | total: 10000 | IN: 16 | OUT: 16

=== ARCH: A_16-32-32-16-32-32-16 hidden_sizes: [32, 32, 16, 32, 32] ===

--- RUN N=128 opt=adamw seed=0 ---
[rule32_steps2_A_16-32-32-16-32-32-16_N128_adamw_seed0_steps1000000] step=       0 train= 48.34% test= 48.74% exact=  0.0/  0.0 loss=0.6923
[rule32_steps2_A_16-32-32-16-32-32-16_N128_adamw_seed0_steps1000000] step=       1 train= 49.56% test= 49.98% exact=  0.0/  0.0 loss=0.6905
[rule32_steps2_A_16-32-32-16-32-32-16_N128_adamw_seed0_steps1000000] step=       2 train= 49.56% test= 49.98% exact=  0.0/  0.0 loss=0.6888
[rule32_steps2_A_16-32-32-16-32-32-16_N128_adamw_seed0_steps1000000] step=       3 train= 49.56% test= 49.98% exact=  0.0/  0.0 loss=0.6870
[rule32_steps2_A_16-32-32-16-32-32-16_

In [8]:
# =========================
# RULE-30 (1-step) GD BASELINES — TRIMMED (FIXED PRINTING)
# Full-batch AdamW only, deterministic, log-spaced logging up to 1e6 steps
# N = 64, 96, 128, 160
# Saves .log files to results_gd_rule30_1step_trimmed/
# =========================

from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd

# ---------- EDIT PATH ----------
REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

RESULTS_DIR = REPO / "results_gd_rule30_1step_trimmed"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---- locate data_cellauto executable ----
hits = list(REPO.rglob("data_cellauto"))
assert hits, "Could not find data_cellauto. Compile boolearn/src first."
data_cellauto = sorted(hits, key=lambda p: len(str(p)))[0]
print("data_cellauto:", data_cellauto)

# ---- generate dataset: rule30, size=16, steps=1, items=10000 ----
ID = "30_1"
DAT = REPO / f"{ID}.dat"
if not DAT.exists():
    cmd = [str(data_cellauto), "3", "30", "16", "1", "10000", ID]
    print("Generating:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found existing:", DAT)

# ---- load .dat (16->16) ----
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
assert IN == 16 and OUT == 16
assert len(X_all) == 10000
print("Loaded", DAT, "total=", len(X_all))

# ---- model: 16-32-32-16 ----
class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# ---- run config (Veit request) ----
N_LIST = [64, 96, 128, 160]
SEED = 0

LR = 1e-3
WD = 1e-4
MAX_STEPS = 1_000_000
LOG_STEPS = log_spaced_steps(MAX_STEPS, 45)

loss_fn = nn.BCEWithLogitsLoss()

def run_one(N_train: int):
    tag = f"rule30_1step_N{N_train}_adamw_seed{SEED}_steps{MAX_STEPS}"
    log_path = RESULTS_DIR / f"{tag}.log"

    # If already exists, print first+last row so you get output immediately
    if log_path.exists():
        print("[SKIP]", log_path.name)
        df = pd.read_csv(log_path, comment="#")
        print("  first:", df.iloc[0].to_dict())
        print("  last :", df.iloc[-1].to_dict())
        return log_path

    torch.manual_seed(SEED)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    with open(log_path, "w") as f:
        f.write("# GD baseline ReLU MLP (full-batch)\n")
        f.write(f"# rule=30, size=16, steps=1, arch=16-32-32-16, N_train={N_train}, opt=adamw, seed={SEED}, lr={LR}, wd={WD}, max_steps={MAX_STEPS}\n")
        f.write("step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr, 0.5)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte, 0.5)

        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

        # IMPORTANT: print at EVERY logged checkpoint (including step 0)
        print(f"[{tag}] step={step:>7d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f} loss={tr_loss:.4f}")

    # Always log+print step 0
    eval_and_log(0)

    model.train()
    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        loss = loss_fn(logits, Ytr)
        loss.backward()
        opt.step()
        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)
    return log_path

for N_train in N_LIST:
    print(f"\n=== RUN rule30_1step | N={N_train} | AdamW | seed={SEED} ===")
    run_one(N_train)

print("\nDONE. Logs saved in:", RESULTS_DIR)

device: cpu
data_cellauto: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto
Found existing: /Users/mkrishan/Documents/boolearn-main/30_1.dat
Loaded /Users/mkrishan/Documents/boolearn-main/30_1.dat total= 10000

=== RUN rule30_1step | N=64 | AdamW | seed=0 ===
[rule30_1step_N64_adamw_seed0_steps1000000] step=      0 train= 50.29% test= 49.49% exact=  0.0/  0.0 loss=0.6960
[rule30_1step_N64_adamw_seed0_steps1000000] step=      1 train= 50.88% test= 49.50% exact=  0.0/  0.0 loss=0.6954
[rule30_1step_N64_adamw_seed0_steps1000000] step=      2 train= 51.07% test= 49.51% exact=  0.0/  0.0 loss=0.6948
[rule30_1step_N64_adamw_seed0_steps1000000] step=      3 train= 50.88% test= 49.51% exact=  0.0/  0.0 loss=0.6943
[rule30_1step_N64_adamw_seed0_steps1000000] step=      4 train= 50.78% test= 49.52% exact=  0.0/  0.0 loss=0.6938
[rule30_1step_N64_adamw_seed0_steps1000000] step=      5 train= 51.46% test= 49.54% exact=  0.0/  0.0 loss=0.6933
[rule30_1step_N64_adamw_seed0_steps100

In [10]:
# =========================
# RULE-30 (2 steps) — GD baseline
# Full-batch AdamW, deterministic
# N = [64, 96, 128, 160]
# steps = 1e6, log-spaced checkpoints + step 0
# =========================

from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ---------- PATHS ----------
REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

RESULTS_DIR = REPO / "results_gd_rule30_2step_smallN"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------- FIND data_cellauto ----------
hits = list(REPO.rglob("data_cellauto"))
assert hits, "Could not find data_cellauto. Compile boolearn/src first."
data_cellauto = sorted(hits, key=lambda p: len(str(p)))[0]
print("data_cellauto:", data_cellauto)

# ---------- GENERATE DATASET ----------
INBITS = 3
RULE   = 30
SIZE   = 16
STEPS  = 2
ITEMS  = 10000
ID     = "30_2"
DAT    = REPO / f"{ID}.dat"

if not DAT.exists():
    cmd = [str(data_cellauto), str(INBITS), str(RULE), str(SIZE),
           str(STEPS), str(ITEMS), ID]
    print("Generating dataset:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found existing:", DAT)

# ---------- LOAD .dat ----------
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    _, IN = map(int, next(it).split())
    _, OUT = map(int, next(it).split())
    X, Y = [], []
    for _ in range(N_total):
        X.append(list(map(int, next(it).split())))
        Y.append(list(map(int, next(it).split())))
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(Y, dtype=torch.float32),
            IN, OUT)

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
assert IN == 16 and OUT == 16 and len(X_all) == 10000
print("Loaded", DAT, "| total =", len(X_all))

# ---------- MODEL ----------
class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

# ---------- METRICS ----------
@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# ---------- LOG-SPACED STEPS ----------
def log_spaced_steps(max_steps, n_points=45):
    steps = np.unique(
        np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int)
    )
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# ---------- RUN CONFIG ----------
N_LIST     = [64, 96, 128, 160]
SEED       = 0
LR         = 1e-3
WD         = 1e-4
MAX_STEPS  = 1_000_000
LOG_STEPS  = log_spaced_steps(MAX_STEPS)

loss_fn = nn.BCEWithLogitsLoss()

def run_one(N_train):
    tag = f"rule30_2step_N{N_train}_adamw_seed{SEED}_steps{MAX_STEPS}"
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return

    torch.manual_seed(SEED)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    with open(log_path, "w") as f:
        f.write("# Rule 30, 2-step | GD baseline | full-batch AdamW\n")
        f.write(f"# N_train={N_train}, lr={LR}, wd={WD}, seed={SEED}, max_steps={MAX_STEPS}\n")
        f.write("step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr)
            te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},"
                    f"{tr_bit:.4f},{te_bit:.4f},"
                    f"{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[N={N_train}] step={step:>7d} "
              f"train={tr_bit:6.2f}% test={te_bit:6.2f}% "
              f"exact={tr_ex:5.1f}/{te_ex:5.1f}")

    # ---- STEP 0 ----
    eval_and_log(0)
    model.train()

    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        opt.step()
        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)

# ---------- RUN ----------
for N in N_LIST:
    print(f"\n=== RUN Rule30-2step | N={N} ===")
    run_one(N)

print("\nALL DONE. Logs in:", RESULTS_DIR)

device: cpu
data_cellauto: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto
Generating dataset: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto 3 30 16 2 10000 30_2
Loaded /Users/mkrishan/Documents/boolearn-main/30_2.dat | total = 10000

=== RUN Rule30-2step | N=64 ===
[N=64] step=      0 train= 51.56% test= 49.92% exact=  0.0/  0.0
[N=64] step=      1 train= 51.66% test= 50.02% exact=  0.0/  0.0
[N=64] step=      2 train= 52.34% test= 50.11% exact=  0.0/  0.0
[N=64] step=      3 train= 52.34% test= 50.23% exact=  0.0/  0.0
[N=64] step=      4 train= 53.32% test= 50.30% exact=  0.0/  0.0
[N=64] step=      5 train= 53.71% test= 50.44% exact=  0.0/  0.0
[N=64] step=      7 train= 54.39% test= 50.54% exact=  0.0/  0.0
[N=64] step=      9 train= 55.37% test= 50.60% exact=  0.0/  0.0
[N=64] step=     12 train= 57.32% test= 50.63% exact=  0.0/  0.0
[N=64] step=     17 train= 58.59% test= 50.49% exact=  0.0/  0.0
[N=64] step=     23 train= 57.81% test= 50.

In [12]:
from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ----- paths -----
REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----- find data_cellauto -----
hits = list(REPO.rglob("data_cellauto"))
assert hits, "Could not find data_cellauto (compile boolearn/src first)."
data_cellauto = sorted(hits, key=lambda p: len(str(p)))[0]

# ----- generate rule30_1 dataset (if needed) -----
DAT = REPO / "30_1.dat"
if not DAT.exists():
    cmd = [str(data_cellauto), "3", "30", "16", "1", "10000", "30_1"]
    print("Generating:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found:", DAT)

# ----- load boolearn .dat -----
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    _, IN = map(int, next(it).split())
    _, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
assert IN == 16 and OUT == 16 and len(X_all) == 10000
print("Loaded:", DAT, "total=", len(X_all))

# ----- model: 16-32-32-16 -----
class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

# ----- metrics -----
@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# ----- log schedule -----
def log_spaced_steps(max_steps: int, n_points: int = 25):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

device: cpu
Found: /Users/mkrishan/Documents/boolearn-main/30_1.dat
Loaded: /Users/mkrishan/Documents/boolearn-main/30_1.dat total= 10000


In [14]:
@torch.no_grad()
def project_rows_topk_norm(W: torch.Tensor, k: int, target_norm2: float):
    """
    Row-wise projection:
      1) keep top-k |weights|
      2) zero rest
      3) rescale each row to ||row||^2 = target_norm2
    """
    if W.ndim != 2:
        return

    absW = W.abs()
    idx = absW.topk(k, dim=1, largest=True).indices  # (out_dim, k)
    mask = torch.zeros_like(W, dtype=torch.bool)
    mask.scatter_(1, idx, True)

    W.mul_(mask)

    row_norm2 = (W * W).sum(dim=1, keepdim=True).clamp_min(1e-12)
    scale = (target_norm2 / row_norm2).sqrt()
    W.mul_(scale)

@torch.no_grad()
def project_model_topk_norm(model: nn.Module, k: int = 3):
    """
    Apply projection to every Linear layer weights.
    Target row norm squared = fan_in (matches your BTF normalization idea).
    """
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            fan_in = mod.weight.shape[1]
            project_rows_topk_norm(mod.weight, k=k, target_norm2=float(fan_in))

def run_projected_adamw_smallN(
    N_train=64,
    max_steps=20000,
    log_points=25,
    lr=3e-4,               # lower lr is usually safer with projections
    wd=1e-4,
    k=3,
    seed=0,
):
    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    LOG_STEPS = log_spaced_steps(max_steps, log_points)

    def eval_print(step):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        print(f"[Proj-AdamW N={N_train}] step={step:>6d} "
              f"train={tr_bit:6.2f}% test={te_bit:6.2f}% "
              f"exact={tr_ex:5.1f}/{te_ex:5.1f} loss={tr_loss:.4f}")

    eval_print(0)
    model.train()
    for step in range(1, max_steps + 1):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        opt.step()

        # HARD projection (this is the key)
        project_model_topk_norm(model, k=k)

        if step in LOG_STEPS:
            eval_print(step)

    return model

# ---- run small-N test ----
_ = run_projected_adamw_smallN(N_train=64, max_steps=20000, k=3)

[Proj-AdamW N=64] step=     0 train= 50.29% test= 49.49% exact=  0.0/  0.0 loss=0.6960
[Proj-AdamW N=64] step=     1 train= 51.27% test= 49.37% exact=  0.0/  0.0 loss=8.2773
[Proj-AdamW N=64] step=     2 train= 51.27% test= 49.37% exact=  0.0/  0.0 loss=8.2741
[Proj-AdamW N=64] step=     3 train= 51.27% test= 49.37% exact=  0.0/  0.0 loss=8.2704
[Proj-AdamW N=64] step=     5 train= 51.27% test= 49.36% exact=  0.0/  0.0 loss=8.2624
[Proj-AdamW N=64] step=     8 train= 51.27% test= 49.36% exact=  0.0/  0.0 loss=8.2498
[Proj-AdamW N=64] step=    12 train= 51.37% test= 49.38% exact=  0.0/  0.0 loss=8.2328
[Proj-AdamW N=64] step=    18 train= 51.37% test= 49.39% exact=  0.0/  0.0 loss=8.2072
[Proj-AdamW N=64] step=    27 train= 51.37% test= 49.49% exact=  0.0/  0.0 loss=8.1688
[Proj-AdamW N=64] step=    41 train= 51.56% test= 49.50% exact=  0.0/  0.0 loss=8.1096
[Proj-AdamW N=64] step=    62 train= 51.27% test= 49.44% exact=  0.0/  0.0 loss=8.0232
[Proj-AdamW N=64] step=    94 train= 51.27%

In [16]:
def norm_penalty_per_row(W: torch.Tensor, target_norm2: float):
    """
    Sum over rows: (||row||^2 - target_norm2)^2
    """
    row_norm2 = (W * W).sum(dim=1)
    return ((row_norm2 - target_norm2) ** 2).mean()

def l1_penalty(W: torch.Tensor):
    return W.abs().mean()

def run_penalty_adamw_smallN(
    N_train=64,
    max_steps=20000,
    log_points=25,
    lr=1e-3,
    wd=1e-4,
    lam_norm=1e-2,      # tune
    lam_l1=1e-4,        # tune
    seed=0,
):
    torch.manual_seed(seed)

    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    LOG_STEPS = log_spaced_steps(max_steps, log_points)

    def structured_penalty(model: nn.Module):
        pen = 0.0
        for mod in model.modules():
            if isinstance(mod, nn.Linear):
                fan_in = mod.weight.shape[1]
                pen = pen + norm_penalty_per_row(mod.weight, target_norm2=float(fan_in))
                pen = pen + lam_l1 * l1_penalty(mod.weight)
        return lam_norm * pen

    def eval_print(step):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        print(f"[Penalty-AdamW N={N_train}] step={step:>6d} "
              f"train={tr_bit:6.2f}% test={te_bit:6.2f}% "
              f"exact={tr_ex:5.1f}/{te_ex:5.1f} base_loss={tr_loss:.4f}")

    eval_print(0)
    model.train()
    for step in range(1, max_steps + 1):
        opt.zero_grad()
        logits = model(Xtr)
        base = loss_fn(logits, Ytr)
        pen = structured_penalty(model)
        loss = base + pen
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_print(step)

    return model

# ---- run small-N test ----
_ = run_penalty_adamw_smallN(N_train=64, max_steps=20000, lam_norm=1e-2, lam_l1=1e-4)

[Penalty-AdamW N=64] step=     0 train= 50.29% test= 49.49% exact=  0.0/  0.0 base_loss=0.6960
[Penalty-AdamW N=64] step=     1 train= 50.39% test= 49.47% exact=  0.0/  0.0 base_loss=0.6959
[Penalty-AdamW N=64] step=     2 train= 50.39% test= 49.45% exact=  0.0/  0.0 base_loss=0.6958
[Penalty-AdamW N=64] step=     3 train= 50.20% test= 49.42% exact=  0.0/  0.0 base_loss=0.6957
[Penalty-AdamW N=64] step=     5 train= 50.20% test= 49.37% exact=  0.0/  0.0 base_loss=0.6955
[Penalty-AdamW N=64] step=     8 train= 50.00% test= 49.34% exact=  0.0/  0.0 base_loss=0.6952
[Penalty-AdamW N=64] step=    12 train= 50.59% test= 49.32% exact=  0.0/  0.0 base_loss=0.6948
[Penalty-AdamW N=64] step=    18 train= 50.98% test= 49.37% exact=  0.0/  0.0 base_loss=0.6943
[Penalty-AdamW N=64] step=    27 train= 50.78% test= 49.59% exact=  0.0/  0.0 base_loss=0.6934
[Penalty-AdamW N=64] step=    41 train= 50.59% test= 49.99% exact=  0.0/  0.0 base_loss=0.6922
[Penalty-AdamW N=64] step=    62 train= 51.95% tes

In [18]:
from pathlib import Path
import subprocess
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Find data_cellauto
hits = list(REPO.rglob("data_cellauto"))
assert hits, "Could not find data_cellauto (compile boolearn/src first)."
data_cellauto = sorted(hits, key=lambda p: len(str(p)))[0]

# Generate rule30_1 dataset if needed: 30_1.dat (size=16, steps=1, items=10000)
DAT = REPO / "30_1.dat"
if not DAT.exists():
    cmd = [str(data_cellauto), "3", "30", "16", "1", "10000", "30_1"]
    print("Generating:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(REPO), check=True)
else:
    print("Found:", DAT)

def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    _, IN = map(int, next(it).split())
    _, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    return X, Y, IN, OUT

X_all, Y_all, IN, OUT = load_dat_bits(DAT)
assert IN == 16 and OUT == 16 and len(X_all) == 10000
print("Loaded:", DAT, "total=", len(X_all))

class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def log_spaced_steps(max_steps: int, n_points: int = 45):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

# Run grid
N_LIST = [64, 96, 128, 160]
SEED = 0
MAX_STEPS = 1_000_000
LOG_STEPS = log_spaced_steps(MAX_STEPS, 45)
loss_fn = nn.BCEWithLogitsLoss()

device: cpu
Found: /Users/mkrishan/Documents/boolearn-main/30_1.dat
Loaded: /Users/mkrishan/Documents/boolearn-main/30_1.dat total= 10000


In [20]:
RESULTS_DIR = REPO / "results_rule30_proj_adamw"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

@torch.no_grad()
def project_rows_topk_norm(W: torch.Tensor, k: int, target_norm2: float):
    if W.ndim != 2:
        return
    absW = W.abs()
    idx = absW.topk(k, dim=1, largest=True).indices
    mask = torch.zeros_like(W, dtype=torch.bool)
    mask.scatter_(1, idx, True)
    W.mul_(mask)
    row_norm2 = (W * W).sum(dim=1, keepdim=True).clamp_min(1e-12)
    scale = (target_norm2 / row_norm2).sqrt()
    W.mul_(scale)

@torch.no_grad()
def project_model_topk_norm(model: nn.Module, k: int = 3):
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            fan_in = mod.weight.shape[1]
            project_rows_topk_norm(mod.weight, k=k, target_norm2=float(fan_in))

def run_proj_adamw(N_train: int, k: int = 3, lr: float = 3e-4, wd: float = 1e-4):
    tag = f"rule30_1step_projAdamW_k{k}_N{N_train}_seed{SEED}_steps{MAX_STEPS}"
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return log_path

    torch.manual_seed(SEED)

    Xtr = X_all[:N_train].to(device); Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device); Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    with open(log_path, "w") as f:
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[ProjAdamW N={N_train}] step={step:>7d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

    eval_and_log(0)
    model.train()

    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), Ytr)
        loss.backward()
        opt.step()

        # hard projection each step
        project_model_topk_norm(model, k=k)

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)
    return log_path

for N in N_LIST:
    print(f"\n=== ProjAdamW run N={N} ===")
    run_proj_adamw(N_train=N, k=3, lr=3e-4, wd=1e-4)

print("\nDONE. Logs in:", RESULTS_DIR)


=== ProjAdamW run N=64 ===
[ProjAdamW N=64] step=      0 train= 50.29% test= 49.49% exact=  0.0/  0.0
[ProjAdamW N=64] step=      1 train= 51.27% test= 49.37% exact=  0.0/  0.0
[ProjAdamW N=64] step=      2 train= 51.27% test= 49.37% exact=  0.0/  0.0
[ProjAdamW N=64] step=      3 train= 51.27% test= 49.37% exact=  0.0/  0.0
[ProjAdamW N=64] step=      4 train= 51.27% test= 49.36% exact=  0.0/  0.0
[ProjAdamW N=64] step=      5 train= 51.27% test= 49.36% exact=  0.0/  0.0
[ProjAdamW N=64] step=      7 train= 51.27% test= 49.36% exact=  0.0/  0.0
[ProjAdamW N=64] step=      9 train= 51.37% test= 49.38% exact=  0.0/  0.0
[ProjAdamW N=64] step=     12 train= 51.37% test= 49.38% exact=  0.0/  0.0
[ProjAdamW N=64] step=     17 train= 51.37% test= 49.39% exact=  0.0/  0.0
[ProjAdamW N=64] step=     23 train= 51.37% test= 49.48% exact=  0.0/  0.0
[ProjAdamW N=64] step=     32 train= 51.37% test= 49.54% exact=  0.0/  0.0
[ProjAdamW N=64] step=     43 train= 51.56% test= 49.50% exact=  0.0/  0

In [21]:
RESULTS_DIR = REPO / "results_rule30_penalty_adamw"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def norm_penalty_rows(W: torch.Tensor, target_norm2: float):
    row_norm2 = (W * W).sum(dim=1)
    return ((row_norm2 - target_norm2) ** 2).mean()

def l1_penalty(W: torch.Tensor):
    return W.abs().mean()

def structured_penalty(model: nn.Module, lam_norm: float, lam_l1: float):
    pen = 0.0
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            fan_in = mod.weight.shape[1]
            pen = pen + norm_penalty_rows(mod.weight, target_norm2=float(fan_in))
            pen = pen + l1_penalty(mod.weight)
    return lam_norm * pen + lam_l1 * 0.0  # keep lam_l1 separate below if you prefer

def run_penalty_adamw(N_train: int, lr: float = 1e-3, wd: float = 1e-4, lam_norm: float = 1e-2, lam_l1: float = 1e-4):
    tag = f"rule30_1step_penaltyAdamW_N{N_train}_seed{SEED}_steps{MAX_STEPS}_lamN{lam_norm}_lamL1{lam_l1}"
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return log_path

    torch.manual_seed(SEED)

    Xtr = X_all[:N_train].to(device); Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device); Yte = Y_all[N_train:].to(device)

    model = MLP16_32_32_16().to(device)
    opt = optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    with open(log_path, "w") as f:
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr); te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)
        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")
        print(f"[PenaltyAdamW N={N_train}] step={step:>7d} train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

    eval_and_log(0)
    model.train()

    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad()
        logits = model(Xtr)
        base = loss_fn(logits, Ytr)

        # add penalties
        pen = 0.0
        for mod in model.modules():
            if isinstance(mod, nn.Linear):
                fan_in = mod.weight.shape[1]
                pen = pen + lam_norm * norm_penalty_rows(mod.weight, target_norm2=float(fan_in))
                pen = pen + lam_l1 * l1_penalty(mod.weight)

        loss = base + pen
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)
    return log_path

for N in N_LIST:
    print(f"\n=== PenaltyAdamW run N={N} ===")
    run_penalty_adamw(N_train=N, lr=1e-3, wd=1e-4, lam_norm=1e-2, lam_l1=1e-4)

print("\nDONE. Logs in:", RESULTS_DIR)


=== PenaltyAdamW run N=64 ===
[PenaltyAdamW N=64] step=      0 train= 50.29% test= 49.49% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      1 train= 50.39% test= 49.47% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      2 train= 50.39% test= 49.45% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      3 train= 50.20% test= 49.42% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      4 train= 50.29% test= 49.39% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      5 train= 50.20% test= 49.37% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      7 train= 50.10% test= 49.34% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=      9 train= 50.39% test= 49.33% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=     12 train= 50.59% test= 49.32% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=     17 train= 50.68% test= 49.34% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=     23 train= 50.68% test= 49.47% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=     32 train= 50.78% test= 49.70% exact=  0.0/  0.0
[PenaltyAdamW N=64] step=     43 

In [10]:
# =========================
# EXP 1: Random-Logic (m5k1..m5k4) — Penalty AdamW, full-batch, log-spaced
# N = 128/256/384/512, steps=1e6
# Saves .log files to: results_penalty_m5k/
# =========================

from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# ---------- EDIT PATHS ----------
REPO = Path("/Users/mkrishan/Documents/boolearn-main")
assert REPO.exists(), REPO

RESULTS_DIR = REPO / "results_penalty_m5k"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "m5k1": {"dat": REPO / "m5k1.dat", "N_train": 128},
    "m5k2": {"dat": REPO / "m5k2.dat", "N_train": 256},
    "m5k3": {"dat": REPO / "m5k3.dat", "N_train": 384},
    "m5k4": {"dat": REPO / "m5k4.dat", "N_train": 512},
}
for k, cfg in DATASETS.items():
    assert cfg["dat"].exists(), f"Missing {cfg['dat']}"
    print(k, "->", cfg["dat"], "| N_train =", cfg["N_train"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ---------- CONFIG ----------
DEPTH = 5
SEED = 0  # extend to [0,1,2] later if needed

LR = 1e-3
WD = 1e-4

MAX_STEPS = 1_000_000
LOG_POINTS = 45

# Penalty strengths (same spirit as your earlier penalty AdamW)
LAM_NORM = 1e-2
LAM_L1   = 1e-4

# ---------- Helpers ----------
def load_dat_bits(path: Path):
    with open(path, "r") as f:
        lines = [l.strip() for l in f if l.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X_list, Y_list = [], []
    for _ in range(N_total):
        X_list.append(list(map(int, next(it).split())))
        Y_list.append(list(map(int, next(it).split())))
    X = torch.tensor(X_list, dtype=torch.float32)
    Y = torch.tensor(Y_list, dtype=torch.float32)
    assert IN == 32 and OUT == 32, (IN, OUT)
    assert len(X) == N_total
    return X, Y

class MLP32(nn.Module):
    def __init__(self, in_dim=32, depth=5, out_dim=32):
        super().__init__()
        hidden = 32
        layers = [nn.Linear(in_dim, hidden), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Linear(hidden, hidden), nn.ReLU(inplace=True)]
        layers += [nn.Linear(hidden, out_dim)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def log_spaced_steps(max_steps: int, n_points: int):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

LOG_STEPS = log_spaced_steps(MAX_STEPS, LOG_POINTS)
print("Logging at", len(LOG_STEPS), "log-spaced checkpoints (incl 0).")

loss_fn = nn.BCEWithLogitsLoss()

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

def norm_penalty_rows(W: torch.Tensor, target_norm2: float):
    row_norm2 = (W * W).sum(dim=1)
    return ((row_norm2 - target_norm2) ** 2).mean()

def l1_penalty(W: torch.Tensor):
    return W.abs().mean()

def structured_penalty(model: nn.Module, lam_norm: float, lam_l1: float):
    pen_norm = 0.0
    pen_l1 = 0.0
    for mod in model.modules():
        if isinstance(mod, nn.Linear):
            fan_in = mod.weight.shape[1]
            pen_norm = pen_norm + norm_penalty_rows(mod.weight, target_norm2=float(fan_in))
            pen_l1 = pen_l1 + l1_penalty(mod.weight)
    return lam_norm * pen_norm + lam_l1 * pen_l1

def run_id(dataset_key, N_train):
    return f"{dataset_key}_PenaltyAdamW_N{N_train}_seed{SEED}_steps{MAX_STEPS}_lamN{LAM_NORM}_lamL1{LAM_L1}"

def run_one(dataset_key: str, dat_path: Path, N_train: int):
    tag = run_id(dataset_key, N_train)
    log_path = RESULTS_DIR / f"{tag}.log"
    if log_path.exists():
        print("[SKIP]", log_path.name)
        return

    torch.manual_seed(SEED)

    X_all, Y_all = load_dat_bits(dat_path)
    assert len(X_all) == 10000, f"{dataset_key}: expected 10000 total"
    Xtr = X_all[:N_train].to(device)
    Ytr = Y_all[:N_train].to(device)
    Xte = X_all[N_train:].to(device)
    Yte = Y_all[N_train:].to(device)

    model = MLP32(depth=DEPTH).to(device)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    with open(log_path, "w") as f:
        f.write("# Penalty AdamW baseline (full-batch)\n")
        f.write(f"# dataset={dataset_key}, depth={DEPTH}, N_train={N_train}, seed={SEED}, lr={LR}, wd={WD}, "
                f"lam_norm={LAM_NORM}, lam_l1={LAM_L1}, max_steps={MAX_STEPS}\n")
        f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

    def eval_and_log(step: int):
        model.eval()
        with torch.no_grad():
            tr_logits = model(Xtr)
            te_logits = model(Xte)
            tr_loss = loss_fn(tr_logits, Ytr).item()
            te_loss = loss_fn(te_logits, Yte).item()
            tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
            te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)

        with open(log_path, "a") as f:
            f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

        print(f"[{dataset_key} N={N_train}] step={step:>7d} "
              f"train={tr_bit:6.2f}% test={te_bit:6.2f}% exact={tr_ex:5.1f}/{te_ex:5.1f}")

        model.train()

    # step 0
    eval_and_log(0)

    model.train()
    for step in range(1, MAX_STEPS + 1):
        opt.zero_grad(set_to_none=True)
        logits = model(Xtr)
        base = loss_fn(logits, Ytr)
        pen = structured_penalty(model, LAM_NORM, LAM_L1)
        loss = base + pen
        loss.backward()
        opt.step()

        if step in LOG_STEPS:
            eval_and_log(step)

    print("Saved ->", log_path)

# ---------- RUN ----------
for dataset_key, cfg in DATASETS.items():
    print(f"\n=== RUN {dataset_key} | N={cfg['N_train']} ===")
    run_one(dataset_key, cfg["dat"], cfg["N_train"])

print("\nDONE. Logs in:", RESULTS_DIR)

AssertionError: /Users/mkrishan/Documents/boolearn-main